In [13]:
import sys
import time
import threading
import numpy as np
import sounddevice as sd
from scipy.signal import chirp

# ==========================================================
# CONFIGURACIÓN
# ==========================================================
FS = 48000
DURACION_SWEEP = 5.0

F_INICIO = 50
F_FIN = 1500
AMPLITUD = 0.5

MIC = 0        # Canal del micrófono de entrada (0 = Canal 1)
ENTRADA = 1    # ID MiniFuse Entrada
SALIDA = 7     # ID MiniFuse Salida

# ==========================================================
# GENERAR SEÑAL DE SWEEP
# ==========================================================
t = np.linspace(0, DURACION_SWEEP, int(FS * DURACION_SWEEP), endpoint=False)
barrido = AMPLITUD * chirp(t, f0=F_INICIO, t1=DURACION_SWEEP, f1=F_FIN, method="logarithmic")

# Fade in y fade out
fade = int(FS * 0.05)
barrido[:fade] *= np.linspace(0, 1, fade)
barrido[-fade:] *= np.linspace(1, 0, fade)

barrido_estereo = np.column_stack((barrido, barrido)).astype(np.float32)

# Variables globales para monitoreo
rms_actual = -100.0
pico_actual = 0.0
ejecutando = True

# ==========================================================
# CALLBACK DE CAPTURA DE MICRÓFONO
# ==========================================================
def input_callback(indata, frames, time_info, status):
    global rms_actual, pico_actual
    if status:
        print(status, file=sys.stderr)
    
    mic_signal = indata[:, MIC]
    pico = np.max(np.abs(mic_signal))
    rms = np.sqrt(np.mean(mic_signal ** 2))
    
    pico_actual = pico
    rms_actual = 20 * np.log10(rms + 1e-10)

# ==========================================================
# MONITOR EN TIEMPO REAL
# ==========================================================
def mostrar_medidor():
    while ejecutando:
        if pico_actual >= 0.99:
            estado = "SATURACIÓN -> ¡BAJA GANANCIA!"
        elif rms_actual < -18:
            estado = "NIVEL BAJO -> SUBE GANANCIA"
        elif rms_actual > -12:
            estado = "NIVEL ALTO -> BAJA GANANCIA"
        else:
            estado = "CORRECTO (-18 a -12 dBFS)"

        sys.stdout.write(
            f"\rRMS: {rms_actual:6.2f} dBFS | Pico: {pico_actual:5.3f} | Estado: {estado:<30}"
        )
        sys.stdout.flush()
        time.sleep(0.08)

# ==========================================================
# BUCLE PRINCIPAL DE CALIBRACIÓN
# ==========================================================
print("==========================================")
print("       CALIBRACIÓN DE GANANCIA")
print("==========================================")
print(f"Sweep continuo: {F_INICIO} - {F_FIN} Hz")
print("Rango objetivo: -18 dBFS a -12 dBFS")
print("------------------------------------------")

# 1. Reproducir Sweep en bucle continuo (Salida)
sd.play(barrido_estereo, samplerate=FS, device=SALIDA, loop=True)

# 2. Configurar captura en tiempo real (Entrada)
stream_in = sd.InputStream(
    device=ENTRADA,
    channels=2,
    samplerate=FS,
    callback=input_callback
)

hilo_monitor = threading.Thread(target=mostrar_medidor, daemon=True)

try:
    with stream_in:
        hilo_monitor.start()
        print("\nSweep continuo activo.")
        print("Ajusta la perilla en la MiniFuse 2.")
        print("Presiona [ENTER] cuando la ganancia esté calibrada...\n")
        input()

except Exception as e:
    print(f"\nError al abrir dispositivos de audio: {e}")
finally:
    ejecutando = False
    sd.stop()  # Detiene la reproducción continua del sweep
    time.sleep(0.1)

print("\n\n==========================================")
print("        CALIBRACIÓN FINALIZADA")
print("==========================================")
print(f"Nivel final fijado: {rms_actual:.2f} dBFS (Pico: {pico_actual:.3f})")

       CALIBRACIÓN DE GANANCIA
Sweep continuo: 50 - 1500 Hz
Rango objetivo: -18 dBFS a -12 dBFS
------------------------------------------
RMS: -100.00 dBFS | Pico: 0.000 | Estado: NIVEL BAJO -> SUBE GANANCIA   
Sweep continuo activo.
Ajusta la perilla en la MiniFuse 2.
Presiona [ENTER] cuando la ganancia esté calibrada...



RMS: -16.07 dBFS | Pico: 0.226 | Estado: CORRECTO (-18 a -12 dBFS)     

        CALIBRACIÓN FINALIZADA
Nivel final fijado: -15.03 dBFS (Pico: 0.260)
